(introduction-rapids-singlecell)=
# GPU-accelerated analysis with rapids-singlecell

## Motivation

Most chapters in this book run on CPU and on datasets of tens to hundreds of thousands of cells.
Modern atlases now reach the millions, and the same scanpy-style pipeline that finished in minutes on a laptop can take hours on a workstation and fall over on memory before it finishes.

[rapids-singlecell](https://rapids-singlecell.readthedocs.io/) (RSC) is a scverse core package that ports the scanpy API to the GPU on top of [RAPIDS](https://rapids.ai/), CuPy, and cuML {cite}`Dicks2024,Nolet2024`.
It also provides GPU implementations of selected squidpy, decoupler, and pertpy routines.
End-to-end speedups of 10–20× over CPU scanpy are typical for the standard preprocessing and clustering pipeline, with individual steps such as UMAP, t-SNE, and Leiden running 50–100× faster on a single A100 or H100.

This chapter is not a comprehensive tour of the RSC API, but two things at once:

1. A short showcase of how a familiar scanpy pipeline ports to RSC.
2. A focused look at GPU best practices — memory management, host↔device transfers, allocator choice, and out-of-core analysis — because these are what determine whether you actually realize the speedups in practice.

## When does a GPU pay off?

GPUs help when the work is dense, regular, and large enough to amortize the host-to-device transfer cost.
For single-cell data, the rough rule of thumb is:

- **Below ~50k cells**: CPU scanpy is fine, and the transfer overhead can dominate. Use a GPU only if you need to iterate quickly during method development.
- **50k–1M cells on one GPU**: this is the sweet spot. A single 24–80 GB GPU runs the whole standard pipeline interactively.
- **>1M cells, or memory-pressure**: combine RSC with [dask-cuda](https://docs.rapids.ai/api/dask-cuda/stable/) for out-of-core or multi-GPU execution (covered below).

The main bottleneck on a single GPU is **VRAM**, not compute.
A dense float32 expression matrix for 1M cells × 30k genes is ~120 GB — it never fits.
Sparse CSR is what makes single-cell on GPU practical, and most of the best practices below are about keeping the matrix sparse and in the right place for as long as possible.

## Installation

Recent RSC releases ship pre-built wheels with all CUDA dependencies bundled, covering CUDA 12 and 13 and GPU architectures from Turing through Blackwell.
Installation is just:

```bash
pip install rapids-singlecell[rapids13]
```

Use the `[rapids12]` extra on CUDA 12 systems.
On older RSC versions you would have had to install RAPIDS conda packages alongside; that step is no longer needed.

```{admonition} Verify the install
:class: tip, dropdown
Before running anything heavy, confirm CuPy can see your GPU:

    import cupy as cp
    print(cp.cuda.runtime.getDeviceCount(), cp.cuda.Device(0).name)

If this raises `CUDARuntimeError`, your driver version is older than the CUDA runtime in the wheel — update the NVIDIA driver, not the CUDA toolkit.
```

(introduction-rapids-singlecell-key-takeaway-2)=
## Memory management with RMM

RSC routes all GPU allocations through the [RAPIDS Memory Manager](https://github.com/rapidsai/rmm) (RMM) {cite}`rmm2024`.
Importing `rapids_singlecell` already swaps the default CuPy allocator for an RMM-backed one, but the default is the plain CUDA allocator: every allocation is a `cudaMalloc` and every free is a `cudaFree`.
Single-cell pipelines allocate constantly (one temporary array per scanpy function), so this default leaves a lot of performance on the table.

There are two strategies, and you should pick exactly one:

- **Pool allocator** — RMM grabs a large slab of VRAM up front and serves subsequent allocations from it. This is by far the fastest option, but it caps you at what fits in VRAM. Use this whenever your dataset fits.
- **Managed (unified) memory** — allocations are backed by CUDA managed memory, so the driver pages data in and out of host RAM transparently. This is how you analyze a 200 GB dataset on a 40 GB GPU. It is slower per-op (page faults are not free) but still typically beats CPU scanpy.

```{admonition} Do not enable both at the same time
:class: warning
Combining `pool_allocator=True` with `managed_memory=True` looks tempting, but the pool sits on top of managed memory and the two strategies fight each other.
You get the latency cost of paging without the speed of pooling, and fragmentation gets worse over the lifetime of the notebook.
Pick one.
```

Configure the pool allocator at the **very top** of your notebook, before any CuPy array is created.
Reinitializing RMM after CuPy has already allocated will leak the old allocations.

In [ ]:
import cupy as cp
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator

rmm.reinitialize(
    pool_allocator=True,
    managed_memory=False,
    initial_pool_size="4GB",  # start small, grow as needed
    maximum_pool_size="32GB",  # leave headroom for plotting libraries, etc.
)
cp.cuda.set_allocator(rmm_cupy_allocator)

If your dataset will not fit, swap to managed memory instead.
On Linux with a Pascal-or-newer GPU this is true unified memory and oversubscription works out of the box; on Windows under WSL2 it is emulated and somewhat slower.

In [ ]:
rmm.reinitialize(
    pool_allocator=False,
    managed_memory=True,
)
cp.cuda.set_allocator(rmm_cupy_allocator)

With the allocator configured, the rest of the imports look like a normal scanpy notebook — `rapids_singlecell` is conventionally aliased as `rsc`.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import anndata as ad
import scanpy as sc
import pooch
import rapids_singlecell as rsc

(introduction-rapids-singlecell-key-takeaway-1)=
## A scanpy pipeline, on the GPU

RSC mirrors the scanpy module layout on purpose: `rsc.pp` for preprocessing, `rsc.tl` for tools, `rsc.pl` for the few GPU-specific plots, and `rsc.gr` for the squidpy-equivalent neighborhood graph routines.
Most scanpy calls have a one-to-one RSC counterpart with the same signature, so porting a pipeline is largely a matter of switching the prefix.

The mental model is simple:

1. Read the AnnData in as you normally would (CPU memory, sparse `.X`).
2. Move it to the GPU once with `rsc.get.anndata_to_GPU`.
3. Run the heavy preprocessing and clustering steps with `rsc.*`.
4. Move it back to host with `rsc.get.anndata_to_CPU` for plotting, saving, or anything else CPU-only.

We follow the same example dataset used in the upstream RSC tutorials — a ~90k-cell DLI census subset hosted by scverse — so the runtimes here are directly comparable to those reported in the [rapids-singlecell docs](https://rapids-singlecell.readthedocs.io/).

In [ ]:
path = pooch.retrieve(
    url="https://exampledata.scverse.org/rapids-singlecell/dli_census.h5ad",
    fname="dli_census.h5ad"
)
adata = ad.read_h5ad(path)
adata.var_names = adata.var.feature_name
adata

Moving to the GPU converts `.X` (and any layers, by default) to a CuPy CSR matrix in place.
The transfer happens once; everything downstream reads from device memory.

In [ ]:
rsc.get.anndata_to_GPU(adata)

### Quality control

QC metrics, gene filtering, and gene-family flagging all have GPU implementations.
Note that `rsc.pp.flag_gene_family` is the RSC convention for tagging mitochondrial, ribosomal, or other prefix-defined gene sets — it is faster than building a boolean mask in pandas because it does not round-trip `.var_names` to host.

In [ ]:
rsc.pp.flag_gene_family(adata, gene_family_name="MT", gene_family_prefix="MT-")
rsc.pp.flag_gene_family(adata, gene_family_name="RIBO", gene_family_prefix=("RPS", "RPL"))
rsc.pp.calculate_qc_metrics(adata, qc_vars=["MT", "RIBO"])

adata = adata[adata.obs["n_genes_by_counts"] < 5000]
adata = adata[adata.obs["pct_counts_MT"] < 20]
rsc.pp.filter_genes(adata, min_cells=3)
adata.shape

### Normalization, HVG, scaling

These are direct one-to-one ports of the scanpy functions covered in [Normalization](../preprocessing_visualization/normalization.ipynb) and [Feature selection](../preprocessing_visualization/feature_selection.ipynb).
All `flavor=` options for `highly_variable_genes` (`seurat`, `seurat_v3`, `cell_ranger`, `pearson_residuals`) are supported on GPU.

In [ ]:
rsc.pp.normalize_total(adata, target_sum=1e4)
rsc.pp.log1p(adata)
rsc.pp.highly_variable_genes(adata, n_top_genes=5000, flavor="cell_ranger")
adata.raw = adata
rsc.pp.filter_highly_variable(adata)
rsc.pp.scale(adata, max_value=10)

### PCA, neighbors, UMAP, clustering

PCA goes through cuML's randomized SVD; neighbors uses brute-force exact KNN by default, with several approximate methods available for very large datasets.
UMAP and Leiden are also cuML/cuGraph-backed.

In [ ]:
rsc.tl.pca(adata, n_comps=50)
rsc.pp.neighbors(adata, n_neighbors=15, n_pcs=40)
rsc.tl.umap(adata)
rsc.tl.leiden(adata, resolution=0.6)

```{admonition} Approximate KNN trades accuracy for scale
:class: note, dropdown
`rsc.pp.neighbors` exposes the cuVS approximate-nearest-neighbor backends through `algorithm=`:

- `brute` (default): exact KNN, scales as O(n²). Fine up to ~1M cells.
- `ivfflat` / `ivfpq`: inverted-file ANN, scales much better; `ivfpq` adds product quantization to fit very large indices.
- `cagra`: graph-based ANN, the fastest option above ~1M cells and the recommended default at scale.
- `nn_descent`: graph-based, particularly good in very high dimensions.

All of these change cluster assignments at the margin.
If you re-run the same workflow with `brute` and with `cagra`, expect the bulk of cells to land in the same cluster but a few to swap — don't treat that as a bug.
```

### Plotting stays on scanpy

RSC deliberately does not duplicate scanpy's plotting API.
Once the heavy compute is done, move `.X` and the embeddings back to host memory and use the familiar `sc.pl.*` functions.
Keeping the matrix on the GPU after you no longer need it just blocks VRAM that other steps could use, and the plotting libraries (matplotlib, seaborn) live on CPU anyway.

In [ ]:
rsc.get.anndata_to_CPU(adata)
sc.pl.umap(adata, color=["leiden"], legend_loc="on data")

Marker gene scoring with logistic regression has a GPU port that is dramatically faster than the scikit-learn-backed scanpy version on large clusters.
It needs the matrix back on device, which is a useful illustration of how data tends to ping-pong between host and device in a real pipeline.

In [ ]:
rsc.get.anndata_to_GPU(adata)
rsc.tl.rank_genes_groups_logreg(adata, groupby="leiden", use_raw=False)
rsc.get.anndata_to_CPU(adata)

## GPU best practices

The pipeline above runs in seconds on a small dataset. Once you scale up, the habits below are what separate a fast workflow from one that thrashes the GPU.

### Minimize host↔device transfers

Every `anndata_to_GPU` / `anndata_to_CPU` is a PCIe transfer.
A 50 GB sparse matrix takes a few seconds to move and easily dominates fast preprocessing steps.

Move once, do all GPU work in a contiguous block, and move back once.
Avoid patterns like calling a CPU-only function in the middle of a pipeline that forces an implicit move — if you must mix, batch the CPU work.

(introduction-rapids-singlecell-key-takeaway-3)=
### Free intermediates explicitly

Python's garbage collector won't release CuPy memory back to the pool as eagerly as you'd want.
After a memory-heavy step (`rsc.pp.scale` on dense data, `rsc.pp.regress_out`, PCA on a large matrix), drop references to layers you don't need and ask CuPy to release the freed pool blocks back to the driver.

In [ ]:
import gc

# Drop references to large intermediates first.
if "counts" in adata.layers:
    del adata.layers["counts"]
gc.collect()

# Then return any unused pool blocks to the driver.
cp.get_default_memory_pool().free_all_blocks()
cp.get_default_pinned_memory_pool().free_all_blocks()

### Monitor VRAM usage

It's hard to right-size your pool or diagnose an OOM without seeing what's on the device.
There are three useful views:

In [ ]:
# CuPy: what your process has allocated through the active pool.
pool = cp.get_default_memory_pool()
print(f"CuPy pool used: {pool.used_bytes() / 1e9:.2f} GB")
print(f"CuPy pool total: {pool.total_bytes() / 1e9:.2f} GB")

# CUDA: what the device reports as free vs total (across all processes).
free, total = cp.cuda.runtime.memGetInfo()
print(f"Device free: {free / 1e9:.2f} GB / {total / 1e9:.2f} GB")

From a separate terminal, `nvidia-smi --query-gpu=memory.used,memory.free,utilization.gpu --format=csv -l 1` gives you a per-second feed across processes — indispensable when debugging a pipeline that suddenly OOMs three steps in.

### Watch the dtype

AnnData objects coming from `.h5ad` are commonly `float64`.
On the GPU, `float64` doubles your memory footprint and runs noticeably slower on consumer cards (where FP64 throughput is limited compared to data-center cards).

Cast to `float32` before moving to GPU unless you have a specific numerical reason not to:

```python
adata.X = adata.X.astype("float32")
rsc.get.anndata_to_GPU(adata)
```

RSC computes most reductions internally in `float64` to preserve numerical stability for things like variance and PCA, even when the input is `float32`, so you get the memory savings without the worst-case precision loss.

### Plan for non-determinism

GPU reductions are non-deterministic by default because the order of floating-point summation depends on thread scheduling.
Differences are tiny per element but can flip an HVG ranking near the cutoff or move a cell across a cluster boundary.
If exact reproducibility matters, set `cp.random.seed(...)` and constrain to a single CUDA stream, but accept that bit-identical results across hardware are not realistic.

### Match the NVIDIA driver to the CUDA runtime

RSC wheels pin a specific CUDA runtime version.
Your NVIDIA driver must be at least as new as that runtime; an older driver will fail to load CuPy with a misleading error.
Update the driver, not the toolkit — the wheel ships its own toolkit.

### Know what is not ported

Tools that depend heavily on R packages (SoupX, scDblFinder, scran) and tools that aren't bottlenecks (most plotting) live on CPU.
The expected pattern is to do the bottleneck steps on GPU and bounce back to CPU for the rest.

### Watch for MIG on shared clusters

If you're on a shared cluster, check whether MIG (Multi-Instance GPU) is enabled.
MIG slices a single A100/H100 into smaller, isolated GPUs, which is great for fairness but means you may have less VRAM than `nvidia-smi` on the host suggests.

(introduction-rapids-singlecell-key-takeaway-4)=
## Out-of-core analysis with dask-cuda

When the dataset doesn't fit on a single GPU even with managed memory, RSC integrates with [dask-cuda](https://docs.rapids.ai/api/dask-cuda/stable/) to stream chunks of the matrix through one or more GPUs {cite}`Rocklin2015`.
AnnData's lazy reader (`read_elem_lazy` in anndata ≥0.12, `read_elem_as_dask` before that) loads `.X` as a Dask array directly from a Zarr or HDF5 store without ever materializing it in host RAM.

The recipe is short but each piece matters.

In [ ]:
from dask_cuda import LocalCUDACluster
from dask.distributed import Client

cluster = LocalCUDACluster(
    CUDA_VISIBLE_DEVICES="0",
    threads_per_worker=8,
    protocol="ucx",  # zero-copy GPU↔GPU comms over NVLink/InfiniBand if available
    rmm_pool_size="20GB",
    rmm_maximum_pool_size="40GB",
    rmm_allocator_external_lib_list="cupy",
)
client = Client(cluster)

In [ ]:
import zarr
from anndata.experimental import read_elem_lazy

store = zarr.open("my_atlas.zarr")
X_lazy = read_elem_lazy(store["X"], chunks=(20_000, store["X"].attrs["shape"][1]))

adata = ad.AnnData(
    X=X_lazy,
    obs=ad.io.read_elem(store["obs"]),
    var=ad.io.read_elem(store["var"]),
)
rsc.get.anndata_to_GPU(adata)

Two practical points trip up most first-time users of the Dask path:

**Most operations are lazy, but some are not.**
Filtering, `normalize_total`, and `log1p` stay lazy and just queue up work.
`calculate_qc_metrics`, `highly_variable_genes`, `scale`, and `pca` need a full reduction over the matrix and trigger a synchronization — think of those as the natural breakpoints in a Dask pipeline.

**Filter with boolean masks, not `sc.pp.filter_cells`.**
Built-in filtering helpers materialize counts to decide which cells to keep, which defeats lazy execution.
Compute the masks once from QC metrics and apply them directly with `.copy()` to keep things in CSR rather than a view:

```python
adata = adata[adata.obs["n_genes_by_counts"].between(200, 10_000)].copy()
```

**Chunk size is a knob you have to tune.**
Too-small chunks drown the scheduler in tasks; too-large ones blow VRAM on synchronization.
20–50k cells per chunk is a reasonable starting point for a 24–80 GB GPU; profile with the Dask dashboard if a step is slower than expected.

## Beyond preprocessing: decoupler, squidpy, and pertpy on the GPU

RSC also wraps GPU implementations of selected routines from three other scverse packages, each under its own namespace:

- `rsc.dcg.*` — [decoupler](https://decoupler-py.readthedocs.io/) functional analysis (`ulm`, `mlm`, `aucell`, `waggr`, `zscore`). On a million-cell atlas, transcription factor activity inference goes from hours on CPU to minutes on a single GPU. The CPU and GPU APIs are identical aside from the namespace; see the [pathway and gene-set analysis chapter](../conditions/gsea_pathway.ipynb).
- `rsc.gr.*` — [squidpy](https://squidpy.readthedocs.io/) spatial analysis (`spatial_autocorr`, `co_occurrence`, `ligrec`). Moran's/Geary's spatial autocorrelation and the ligand–receptor permutation test are the routines that benefit most from the GPU; see the [spatial neighborhood chapter](../spatial/neighborhood.ipynb) for what these do conceptually. The neighborhood graph itself is still built with `sq.gr.spatial_neighbors` on CPU.
- `rsc.ptg.*` — [pertpy](https://pertpy.readthedocs.io/) perturbation distances via the `Distance` class (`edistance`, `mmd`, and other metrics). The GPU implementation recomputes distances on the fly rather than materializing an n×n cell-distance matrix, so bootstrap variance estimation becomes practical on screens with hundreds of perturbations; see the [perturbation modeling chapter](../conditions/perturbation_modeling.ipynb).

```python
import decoupler as dc
net = dc.op.collectri(organism="human", license="academic")
rsc.dcg.ulm(adata, net=net, raw=False, bsize=10_000)

import squidpy as sq
sq.gr.spatial_neighbors(adata, coord_type="generic")  # still CPU
rsc.gr.spatial_autocorr(adata, mode="moran")

distance = rsc.ptg.Distance(metric="edistance", obsm_key="X_pca")
df = distance.pairwise(adata, groupby="perturbation")
```

Not the entire upstream API is ported — RSC focuses on the routines that are both heavy enough to be worth a GPU port and embarrassingly parallel enough to stay numerically faithful. Check the RSC API reference for the current coverage.

```{seealso}
:class: dropdown
- [rapids-singlecell documentation](https://rapids-singlecell.readthedocs.io/) — API reference and tutorials, including 1M-cell and out-of-core demos.
- [RAPIDS Memory Manager](https://docs.rapids.ai/api/rmm/stable/) — deeper background on pool, managed, and arena allocators.
- [dask-cuda](https://docs.rapids.ai/api/dask-cuda/stable/) — multi-GPU and out-of-core configuration.
- [cuML user guide](https://docs.rapids.ai/api/cuml/stable/) — the GPU ML library that backs PCA, UMAP, t-SNE, and the logistic regression marker test.
- [NVIDIA Single Cell Analysis blog post](https://developer.nvidia.com/blog/gpu-accelerated-single-cell-rna-analysis-with-rapids-singlecell/) — reference benchmarks against CPU scanpy.
```

## Quiz

In [ ]:
%run ../src/lib.py
multiple_choice_question(
    "q1",
    "You have an 80 GB AnnData object and a 40 GB GPU. Which RMM configuration is appropriate?",
    [
        "pool_allocator=True, managed_memory=False",
        "pool_allocator=False, managed_memory=True",
        "pool_allocator=True, managed_memory=True",
        "Don't reinitialize RMM; let the default allocator handle it",
    ],
    "pool_allocator=False, managed_memory=True",
    {
        "pool_allocator=True, managed_memory=False": "A pool allocator caps you at VRAM size. The dataset doesn't fit.",
        "pool_allocator=True, managed_memory=True": "Mixing the two strategies fragments memory and erodes the speedup.",
        "Don't reinitialize RMM; let the default allocator handle it": "The default plain-cudaMalloc allocator works but pays an allocation cost on every temporary array.",
    },
)
multiple_choice_question(
    "q2",
    "Which of the following steps in a Dask-backed RSC pipeline triggers a synchronization (i.e., is not lazy)?",
    [
        "rsc.pp.normalize_total",
        "rsc.pp.log1p",
        "rsc.pp.highly_variable_genes",
        "Boolean indexing into adata",
    ],
    "rsc.pp.highly_variable_genes",
    {
        "rsc.pp.normalize_total": "normalize_total stays lazy.",
        "rsc.pp.log1p": "log1p stays lazy.",
        "Boolean indexing into adata": "Boolean indexing is lazy as long as you avoid sc.pp.filter_cells / .filter_genes.",
    },
)
flip_card(
    "q3",
    "Why should you cast .X to float32 before moving to the GPU?",
    "It halves the VRAM footprint and runs faster on consumer GPUs where FP64 throughput is limited; RSC still uses float64 internally for sensitive reductions.",
)

## References

```{bibliography}
:filter: docname in docnames
```

## Contributors

We gratefully acknowledge the contributions of:

### Authors

* Lukas Heumos

### Reviewers